# 🏭 PM10 Kraków — Prognozowanie Jakości Powietrza
## Sekcja 1: Data Loading & Preprocessing

---
**Stacja docelowa:** `MpKrakAlKras` (Al. Krasińskiego)  
**Stacje pomocnicze:** `MpKrakBujaka`, `MpKrakBulwar`, `MpKrakWadow`  
**Zakres danych:** 2018–2024 (dobowe)  

### ⚠️ Kluczowe zasady preprocessingu (audyt)
| # | Zasada | Uzasadnienie |
|---|--------|--------------|
| 1 | Outliery usuwane **tylko** na zbiorze treningowym | Brak data leakage z kwantyli |
| 2 | Flaga długich luk tworzona **przed** interpolacją | Poprawna logika flag |
| 3 | `ffill/bfill` tylko dla luk > 3 dni, z ostrzeżeniem | Transparentność imputacji |
| 4 | Zakres lat 2018–2024 zgodny z opisem projektu | Spójność danych |

## 📦 1. Import bibliotek

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import seaborn as sns
import calendar
import requests
import holidays
import matplotlib.colors as colors

from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

from scipy import stats
from scipy.stats import boxcox
from scipy.special import inv_boxcox

from datetime import datetime
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import pmdarima as pm
from prophet import Prophet
import lightgbm as lgb

import plotly.graph_objects as go
import plotly.express as px

import warnings
warnings.filterwarnings('ignore')

# Stałe projektu
STATIONS      = ['MpKrakAlKras', 'MpKrakBujaka', 'MpKrakBulwar', 'MpKrakWadow']
TARGET        = 'MpKrakAlKras'
AUX_STATIONS  = ['MpKrakBujaka', 'MpKrakBulwar', 'MpKrakWadow']
TRAIN_END     = '2022-12-31'   # granica train — używana też do outlierów
VAL_END       = '2023-12-31'
# TEST: 2024-01-01 — 2024-12-31

print('✅ Biblioteki załadowane.')
print(f'   TARGET : {TARGET}')
print(f'   TRAIN  : do {TRAIN_END}')
print(f'   VAL    : do {VAL_END}')
print(f'   TEST   : 2024-01-01 — 2024-12-31')

## 📂 2. Wczytywanie danych PM10

> **Zmiana vs oryginał:** zakres `range(2018, 2025)` zamiast `range(2019, 2025)` — dodajemy brakujący rok 2018 zgodnie z opisem projektu.  
> Dodano funkcję `load_pm10_excel()` z odpornym parsingiem nagłówka (bez magicznych liczb `iloc[4]`).

In [ ]:
def load_pm10_excel(year: int) -> pd.DataFrame:
    """
    Wczytuje jeden roczny plik PM10 z odpornym parsingiem.
    Dynamicznie szuka wiersza z nagłówkiem zawierającego 'Kod stacji'
    zamiast hardkodować numer wiersza.
    """
    path = f'../data/{year}_PM10_24g.xlsx'
    df_raw = pd.read_excel(path, header=None)

    # Znajdź wiersz nagłówkowy
    header_row = None
    for i, row in df_raw.iterrows():
        if row.astype(str).str.contains('Kod stacji', na=False).any():
            header_row = i
            break

    if header_row is None:
        raise ValueError(f'Nie znaleziono nagłówka "Kod stacji" w pliku: {path}')

    # Wczytaj ponownie z właściwym nagłówkiem, pomijając wiersze meta po nagłówku
    df = pd.read_excel(path, header=header_row)

    # Usuń wiersze, które są metadanymi (przed faktycznymi danymi)
    # Identyfikujemy je po tym, że kolumna 'Kod stacji' nie jest datą
    df = df.rename(columns={'Kod stacji': 'Date'})
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df = df.dropna(subset=['Date']).reset_index(drop=True)

    print(f'  [{year}] wierszy: {len(df)}')
    return df


# ✅ Poprawiony zakres: 2018–2024 (zgodny z opisem projektu)
years = range(2018, 2025)

print('📂 Wczytywanie plików PM10...')
lista_df = [load_pm10_excel(y) for y in years]
pm10_all = pd.concat(lista_df, ignore_index=True)

print(f'\n✅ Łącznie wierszy: {len(pm10_all)}')
print(f'   Zakres dat     : {pm10_all["Date"].min().date()} → {pm10_all["Date"].max().date()}')
pm10_all.head(3)

## 🔧 3. Selekcja stacji i przygotowanie indeksu

In [ ]:
cols_to_keep = ['Date'] + STATIONS
krak_stations = pm10_all[cols_to_keep].copy()

krak_stations = krak_stations.set_index('Date')

# Konwersja do float (obsługa przecinków jako separatora dziesiętnego)
for col in STATIONS:
    krak_stations[col] = (
        krak_stations[col]
        .astype(str)
        .str.replace(',', '.', regex=False)
        .pipe(pd.to_numeric, errors='coerce')
    )

# Wymuszenie pełnego dziennego indeksu (bez dziur)
krak_stations = krak_stations.asfreq('D')

print('✅ Ramka krak_stations:')
print(f'   Zakres: {krak_stations.index.min().date()} → {krak_stations.index.max().date()}')
print(f'   Kształt: {krak_stations.shape}')
print()
print('Braki danych (%):')
print((krak_stations.isnull().sum() / len(krak_stations) * 100).round(2).to_string())

## 🩹 4. Imputacja braków danych

> **Zmiana vs oryginał:** flaga `_long_gap` tworzona **przed** interpolacją na podstawie faktycznej długości luki (nie warunku po interpolacji). `ffill/bfill` tylko gdy pozostają NaN po interpolacji, z wyraźnym ostrzeżeniem.

In [ ]:
def interpolate_with_gap_flag(df: pd.DataFrame,
                               col: str,
                               limit: int = 3) -> pd.DataFrame:
    """
    Interpoluje krótkie luki (≤ limit dni), flaguje długie luki.
    Flaga tworzona PRZED interpolacją — poprawna logika.

    Parametry
    ----------
    df    : DataFrame z indeksem DatetimeIndex
    col   : nazwa kolumny do interpolacji
    limit : maksymalna długość luki do interpolacji (dni)

    Zwraca
    -------
    DataFrame z uzupełnioną kolumną i flagą `{col}_long_gap`
    """
    df = df.copy()
    is_null = df[col].isnull()

    # --- Krok 1: Zlicz rozmiar każdej ciągłej luki ---
    # ne().cumsum() tworzy unikalne ID dla każdego bloku (null/not-null)
    gap_groups = is_null.ne(is_null.shift()).cumsum()
    gap_sizes  = is_null.groupby(gap_groups).transform('sum')

    # --- Krok 2: Flaga długich luk (tworzona PRZED interpolacją) ---
    df[f'{col}_long_gap'] = ((is_null) & (gap_sizes > limit)).astype(int)

    n_long_days = df[f'{col}_long_gap'].sum()
    n_short_days = (is_null & (gap_sizes <= limit)).sum()
    print(f'  [{col}]  krótkie luki (≤{limit}d): {n_short_days} dni | '
          f'długie luki (>{limit}d): {n_long_days} dni → flaga ustawiona')

    # --- Krok 3: Interpolacja krótkich luk ---
    df[col] = df[col].interpolate(method='time', limit=limit)

    # --- Krok 4: ffill/bfill tylko dla długich luk (z ostrzeżeniem) ---
    remaining = df[col].isnull().sum()
    if remaining > 0:
        print(f'  ⚠️  [{col}] {remaining} dni uzupełnionych ffill/bfill '
              f'(długie luki — rozważ usunięcie tych wierszy z treningu)')
        df[col] = df[col].ffill().bfill()

    return df


print('🩹 Imputacja braków danych...')
for col in STATIONS:
    krak_stations = interpolate_with_gap_flag(krak_stations, col, limit=3)

print()
print('✅ Braki po imputacji:')
print(krak_stations[STATIONS].isnull().sum().to_string())

## 📊 5. Wizualizacja braków i długich luk

In [ ]:
gap_flag_cols = [c for c in krak_stations.columns if '_long_gap' in c]

if gap_flag_cols:
    fig, axes = plt.subplots(len(gap_flag_cols), 1,
                             figsize=(14, 2.5 * len(gap_flag_cols)),
                             sharex=True)
    if len(gap_flag_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, gap_flag_cols):
        station = col.replace('_long_gap', '')
        ax.fill_between(krak_stations.index,
                        krak_stations[col],
                        color='tomato', alpha=0.8, label='Długa luka (>3 dni)')
        ax.set_ylabel(station, fontsize=9)
        ax.set_ylim(-0.1, 1.5)
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('Data')
    fig.suptitle('Długie luki w danych (>3 dni) — imputowane ffill/bfill',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Brak długich luk w żadnej stacji.')

## 🚨 6. Usuwanie outlierów targetu

> **KRYTYCZNA ZMIANA vs oryginał:** kwantyle IQR liczone **wyłącznie na zbiorze treningowym** (`≤ TRAIN_END`).  
> W oryginale były liczone na całym datasecie → **data leakage** (model pośrednio widział rozkład zbioru testowego).
>
> Granica z train jest następnie stosowana do **całego** df (również test) — to jedyne prawidłowe podejście.

In [ ]:
def remove_outliers_train_only(df: pd.DataFrame,
                                col: str,
                                train_end: str,
                                iqr_multiplier: float = 3.0) -> pd.DataFrame:
    """
    Usuwa outliery z kolumny. Granice IQR liczone WYŁĄCZNIE na danych
    treningowych (do train_end włącznie) — brak data leakage.

    Parametry
    ----------
    df             : DataFrame z indeksem DatetimeIndex
    col            : kolumna do czyszczenia
    train_end      : ostatni dzień zbioru treningowego (str, np. '2022-12-31')
    iqr_multiplier : mnożnik IQR dla granic (domyślnie 3.0)
    """
    df = df.copy()
    train_mask = df.index <= train_end

    # Statystyki TYLKO z danych treningowych
    Q1 = df.loc[train_mask, col].quantile(0.25)
    Q3 = df.loc[train_mask, col].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + iqr_multiplier * IQR
    lower_bound = max(0.0, Q1 - iqr_multiplier * IQR)  # PM10 ≥ 0

    n_out_train = ((df.loc[train_mask, col] > upper_bound) |
                   (df.loc[train_mask, col] < lower_bound)).sum()
    n_out_test  = ((df.loc[~train_mask, col] > upper_bound) |
                   (df.loc[~train_mask, col] < lower_bound)).sum()

    print(f'[{col}] Granice (z train): lower={lower_bound:.1f}, upper={upper_bound:.1f}')
    print(f'[{col}] Outlierów w train : {n_out_train}')
    print(f'[{col}] Outlierów w test  : {n_out_test} '
          f'(nie usuwamy, ale maskujemy tą samą granicą)')

    # Maskuj outliery w całym df (też test) — granicą z train
    outlier_mask = (df[col] > upper_bound) | (df[col] < lower_bound)
    df.loc[outlier_mask, col] = np.nan

    # Uzupełnij interpolacją
    df[col] = df[col].interpolate(method='time', limit=3)
    df[col] = df[col].ffill().bfill()

    return df


print('🚨 Usuwanie outlierów (tylko granice z train)...')
krak_stations = remove_outliers_train_only(
    krak_stations, TARGET, TRAIN_END, iqr_multiplier=3.0
)

print(f'\n✅ Braki w {TARGET} po usunięciu outlierów: {krak_stations[TARGET].isnull().sum()}')

### Porównanie rozkładu przed/po usunięciu outlierów

In [ ]:
# Odtwórz surowe wartości do porównania
raw_target = pm10_all.set_index('Date').asfreq('D')[TARGET].astype(str).str.replace(',','.').pipe(pd.to_numeric, errors='coerce')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(raw_target.dropna(), bins=60, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].set_title('PM10 — przed usunięciem outlierów', fontweight='bold')
axes[0].set_xlabel('PM10 [µg/m³]')
axes[0].set_ylabel('Liczba dni')

axes[1].hist(krak_stations[TARGET].dropna(), bins=60, color='seagreen', alpha=0.7, edgecolor='white')
axes[1].set_title('PM10 — po usunięciu outlierów (granica z train)', fontweight='bold')
axes[1].set_xlabel('PM10 [µg/m³]')

plt.tight_layout()
plt.show()

print(f'Mediana przed: {raw_target.median():.1f} µg/m³  |  po: {krak_stations[TARGET].median():.1f} µg/m³')
print(f'Max przed    : {raw_target.max():.1f} µg/m³  |  po: {krak_stations[TARGET].max():.1f} µg/m³')

## 🌤️ 7. Pobieranie danych meteorologicznych

> **Zmiana vs oryginał:** dodano `temperature_2m_min`, `temperature_2m_max`, `wind_speed_10m_mean`, `relative_humidity_2m_mean`, `snowfall_sum` — niezbędne do poprawnego Feature Engineering (HDD, inwersja termiczna, dyspersja pyłów).

In [ ]:
url = 'https://archive-api.open-meteo.com/v1/archive'
params = {
    'latitude'  : 50.0577717,
    'longitude' : 19.9265492,
    'start_date': '2018-01-01',   # ← zgodnie z zakresem projektu
    'end_date'  : '2024-12-31',
    'daily': [
        'temperature_2m_mean',
        'temperature_2m_min',          # ✅ NOWE: do HDD i inwersji termicznej
        'temperature_2m_max',          # ✅ NOWE: amplituda dobowa
        'precipitation_sum',
        'wind_speed_10m_max',
        'wind_speed_10m_mean',         # ✅ NOWE: ważniejsze niż max dla dyspersji
        'surface_pressure_mean',
        'wind_direction_10m_dominant',
        'relative_humidity_2m_mean',   # ✅ NOWE: wilgotność wpływa na PM10
        'snowfall_sum',                # ✅ NOWE: śnieg wymiata pyły
    ],
    'timezone': 'Europe/Berlin'
}

print('🌐 Pobieranie danych meteo z Open-Meteo...')
response = requests.get(url, params=params, timeout=60)
response.raise_for_status()
data = response.json()

weather_df = pd.DataFrame(data['daily'])
weather_df.columns = [
    'date', 'temp_avg', 'temp_min', 'temp_max',
    'rain_sum', 'wind_max', 'wind_mean',
    'pressure_avg', 'wind_dir_dominant',
    'humidity_avg', 'snowfall_sum'
]
weather_df['date'] = pd.to_datetime(weather_df['date'])
weather_df = weather_df.set_index('date')

print(f'✅ Danych meteo: {len(weather_df)} wierszy')
print(f'   Zakres: {weather_df.index.min().date()} → {weather_df.index.max().date()}')
weather_df.head(3)

## 🧭 8. Enkodowanie kierunku wiatru i łączenie danych

In [ ]:
# Enkodowanie kierunku wiatru jako sin/cos (cykliczne)
if weather_df['wind_dir_dominant'].dtype == object:
    DIR_MAP = {
        'N':0,'NNE':22.5,'NE':45,'ENE':67.5,
        'E':90,'ESE':112.5,'SE':135,'SSE':157.5,
        'S':180,'SSW':202.5,'SW':225,'WSW':247.5,
        'W':270,'WNW':292.5,'NW':315,'NNW':337.5
    }
    weather_df['wind_dir_deg'] = weather_df['wind_dir_dominant'].map(DIR_MAP).fillna(0)
else:
    weather_df['wind_dir_deg'] = weather_df['wind_dir_dominant'].fillna(0)

rad = np.deg2rad(weather_df['wind_dir_deg'])
weather_df['wind_dir_sin'] = np.sin(rad)
weather_df['wind_dir_cos'] = np.cos(rad)
weather_df.drop(columns=['wind_dir_dominant', 'wind_dir_deg'], inplace=True)

# Łączenie: PM10 + meteo
df_final = krak_stations.join(weather_df, how='left')

print('✅ Połączono dane PM10 z danymi meteo:')
print(f'   Kształt df_final : {df_final.shape}')
print(f'   Zakres           : {df_final.index.min().date()} → {df_final.index.max().date()}')
print()
print('Braki w kolumnach meteo:')
meteo_cols = [c for c in df_final.columns if c not in STATIONS and '_long_gap' not in c]
missing_meteo = df_final[meteo_cols].isnull().sum()
print(missing_meteo[missing_meteo > 0].to_string() if missing_meteo.any() else '  Brak braków ✅')

## 📋 9. Raport końcowy preprocessingu

In [ ]:
print('=' * 60)
print('        RAPORT KOŃCOWY — DATA PREPROCESSING')
print('=' * 60)
print(f'  Zakres dat      : {df_final.index.min().date()} → {df_final.index.max().date()}')
print(f'  Liczba dni      : {len(df_final)}')
print(f'  Liczba kolumn   : {df_final.shape[1]}')
print()
print('  Podział chronologiczny:')
train_size = (df_final.index <= TRAIN_END).sum()
val_size   = ((df_final.index > TRAIN_END) & (df_final.index <= VAL_END)).sum()
test_size  = (df_final.index > VAL_END).sum()
total      = len(df_final)
print(f'    Train : do {TRAIN_END} → {train_size} dni ({100*train_size/total:.0f}%)')
print(f'    Val   : do {VAL_END}   → {val_size} dni ({100*val_size/total:.0f}%)')
print(f'    Test  : 2024          → {test_size} dni ({100*test_size/total:.0f}%)')
print()
print('  Statystyki targetu (MpKrakAlKras):')
desc = df_final[TARGET].describe()
for stat in ['mean', 'std', 'min', '50%', 'max']:
    print(f'    {stat:>4} : {desc[stat]:.2f} µg/m³')
print()

gap_flags = {c.replace('_long_gap',''):df_final[c].sum()
             for c in df_final.columns if '_long_gap' in c}
print('  Dni z długimi lukami (imputowane ffill/bfill):')
for st, n in gap_flags.items():
    print(f'    {st}: {n} dni')
print('=' * 60)

df_final.head(3)

## 💾 10. Zapis do pliku

Zapisujemy przetworzone dane do pliku Parquet (szybkie wczytywanie w kolejnych notebookach).

In [ ]:
output_path = '../data/df_preprocessed.parquet'
df_final.to_parquet(output_path)
print(f'✅ Dane zapisane do: {output_path}')
print(f'   Kształt: {df_final.shape}')